# Merged 14-Class ECG Classifier (PTB-XL + Chapman)

Trains ECGNetJoint on **60,914 records** (18,524 PTB-XL + 42,390 Chapman-Shaoxing).
Adds **AFIB** (48 → 9,840 samples) and **STACH** (4 → 7,255 samples) to the 12-class model.

---
## Runtime guide

| Cell | Runtime | Duration | What it does |
|------|---------|----------|--------------|
| 1 | **CPU** | 1 min | Mount Drive + install deps |
| 2 | **CPU** | 1 min | Copy scripts from Drive |
| 3 | **CPU** | ~1.5 hrs | Download Chapman from PhysioNet |
| 4 | **CPU** | ~10 min | Build Chapman index + save tar to Drive |
| 5 | **GPU** | 1 min | Switch to GPU — check + mount Drive |
| 6 | **GPU** | ~20 min | Restore Chapman + copy PTB-XL to SSD |
| 7 | **GPU** | ~1-2 hrs | Run merged training |
| 8 | **GPU** | 1 min | Save model to Drive |

**Before running:** Upload these 4 scripts to `MyDrive/EKG_models/`:
`multilabel_merged.py`, `multilabel_classifier.py`, `cnn_classifier.py`, `dataset_chapman.py`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1  ──  CPU runtime
# Mount Drive and install dependencies
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess, sys
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)

import torch
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}  (expected False on CPU runtime)')
print('Cell 1 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2  ──  CPU runtime
# Copy training scripts from Drive to /content/
# ─────────────────────────────────────────────────────────────────────────────
import os, shutil

SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
NEEDED = [
    'multilabel_merged.py',
    'multilabel_classifier.py',
    'cnn_classifier.py',
    'dataset_chapman.py',
]

os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

missing = []
for script in NEEDED:
    src = f'{SCRIPTS_DIR}/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{script}')
        print(f'  Copied {script}')
    else:
        missing.append(script)

if missing:
    raise FileNotFoundError(
        f'Missing scripts on Drive: {missing}\n'
        f'Upload them to {SCRIPTS_DIR}/ first.'
    )
print('Cell 2 done - all scripts copied')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3  ──  CPU runtime
# Download Chapman-Shaoxing from PhysioNet (~1.5 hrs, 16 parallel workers)
# Safe to re-run: skips files already downloaded
# ─────────────────────────────────────────────────────────────────────────────
import os
os.chdir('/content')

CHAPMAN_DIR = '/content/ekg_datasets/chapman'
n_existing = len(list(__import__('pathlib').Path(CHAPMAN_DIR).rglob('*.mat'))) if os.path.exists(CHAPMAN_DIR) else 0
print(f'Existing .mat files: {n_existing} / ~45152')

import dataset_chapman
dataset_chapman.download_chapman(CHAPMAN_DIR)

n_after = len(list(__import__('pathlib').Path(CHAPMAN_DIR).rglob('*.mat')))
print(f'Download complete: {n_after} .mat files')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4  ──  CPU runtime
# Build Chapman index  +  save tar archive to Drive (for GPU session restore)
# Index build: ~5-10 min
# Tar + Drive copy: ~20-30 min (one-time — skipped if tar already on Drive)
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, shutil
os.chdir('/content')

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

# Step 1: Build index
INDEX_PATH = '/content/ekg_datasets/chapman_index.csv'
if not os.path.exists(INDEX_PATH):
    print('Building Chapman index (~5-10 min)...')
    import dataset_chapman
    dataset_chapman.build_chapman_index(
        '/content/ekg_datasets/chapman',
        INDEX_PATH
    )
else:
    print(f'Index already exists: {INDEX_PATH}')

# Step 2: Save Chapman tar to Drive (skip if already saved)
DRIVE_TAR = f'{DRIVE_MODELS}/chapman.tar'
if os.path.exists(DRIVE_TAR):
    size_gb = os.path.getsize(DRIVE_TAR) / 1e9
    print(f'chapman.tar already on Drive ({size_gb:.1f} GB) - skipping')
else:
    print('Creating chapman.tar (~5.5 GB, this takes a few minutes)...')
    subprocess.run(
        ['tar', 'cf', '/content/chapman.tar',
         '-C', '/content',
         'ekg_datasets/chapman',
         'ekg_datasets/chapman_index.csv'],
        check=True
    )
    size_gb = os.path.getsize('/content/chapman.tar') / 1e9
    print(f'Tar created: {size_gb:.1f} GB  -  copying to Drive (~20 min)...')
    shutil.copy('/content/chapman.tar', DRIVE_TAR)
    print(f'Saved to Drive: {DRIVE_TAR}')

print('\nCell 4 done.')
print('>>> NEXT STEP: Switch to GPU runtime (Runtime > Change runtime type > T4 GPU > Save)')
print('>>> After restart, run Cells 5, 6, 7, 8')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5  ──  GPU runtime  (run FIRST after switching to GPU)
# Verify GPU, mount Drive, install dependencies, copy scripts
# ─────────────────────────────────────────────────────────────────────────────
import torch, sys
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to Runtime > Change runtime type > T4 GPU > Save'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)

import os, shutil
SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

for script in ['multilabel_merged.py', 'multilabel_classifier.py',
               'cnn_classifier.py', 'dataset_chapman.py']:
    shutil.copy(f'{SCRIPTS_DIR}/{script}', f'/content/{script}')
    print(f'  Copied {script}')

print('Cell 5 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6  ──  GPU runtime
# Restore Chapman from Drive tar + copy PTB-XL to local SSD
# Chapman restore: ~5 min (tar extract from Drive)
# PTB-XL copy:    ~10-15 min (from Drive to SSD)
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, shutil
os.chdir('/content')

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'

# ── Restore Chapman ───────────────────────────────────────────────────────────
CHAPMAN_INDEX = '/content/ekg_datasets/chapman_index.csv'
if os.path.exists(CHAPMAN_INDEX):
    import pandas as pd
    n = len(pd.read_csv(CHAPMAN_INDEX))
    print(f'Chapman already present: {n} records - skipping restore')
else:
    DRIVE_TAR = f'{DRIVE_MODELS}/chapman.tar'
    if not os.path.exists(DRIVE_TAR):
        raise FileNotFoundError(
            f'Chapman tar not found: {DRIVE_TAR}\n'
            'Re-run the CPU cells (Cells 1-4) first to create it.'
        )
    print(f'Restoring Chapman from {DRIVE_TAR} ...')
    subprocess.run(['tar', 'xf', DRIVE_TAR, '-C', '/content'], check=True)
    import pandas as pd
    n = len(pd.read_csv(CHAPMAN_INDEX))
    print(f'Chapman restored: {n} records')

# ── Copy PTB-XL to local SSD ──────────────────────────────────────────────────
PTBXL_LOCAL = '/content/ekg_datasets/ptbxl'
DRIVE_PTBXL = f'{DRIVE_MODELS}/ptbxl'

if not os.path.exists(DRIVE_PTBXL):
    raise FileNotFoundError(
        f'PTB-XL not found on Drive: {DRIVE_PTBXL}\n'
        'Run Cell 4 in ecgfm_colab_stage2.ipynb to download it first.'
    )

if os.path.exists(PTBXL_LOCAL):
    n_dat = len(list(__import__('pathlib').Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'PTB-XL already on SSD: {n_dat} .dat files - skipping copy')
else:
    n_on_drive = len(list(__import__('pathlib').Path(DRIVE_PTBXL).rglob('*.dat')))
    print(f'Copying PTB-XL from Drive to SSD ({n_on_drive} .dat files, ~10-15 min)...')
    shutil.copytree(DRIVE_PTBXL, PTBXL_LOCAL)
    n_dat = len(list(__import__('pathlib').Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'PTB-XL copied: {n_dat} .dat files on SSD')
    if n_dat < 18000:
        print(f'WARNING: only {n_dat}/21799 .dat files. Training will skip missing records.')
        print('The model will still train but on a smaller PTB-XL subset.')

print('\nCell 6 done - data ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7  ──  GPU runtime
# Run merged 14-class training
# Expected: ~1-2 hrs on T4 (vs 15-25 hrs on CPU)
# Prints per-epoch: Loss | ValAUROC | ValF1
# Saves best model automatically to /content/models/ecg_multilabel_v2.pt
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys
os.chdir('/content')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Run Cell 5 first or switch runtime to T4.')

result = subprocess.run(
    [sys.executable, '-u', 'multilabel_merged.py'],
    check=False
)

if result.returncode != 0:
    print(f'\nTraining exited with code {result.returncode}')
else:
    print('\nTraining complete!')
    model_path = '/content/models/ecg_multilabel_v2.pt'
    if os.path.exists(model_path):
        size_mb = os.path.getsize(model_path) / 1e6
        print(f'Model saved: {model_path} ({size_mb:.1f} MB)')
    else:
        print('WARNING: Model file not found at expected path')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8  ──  GPU runtime
# Save trained model to Drive
# ─────────────────────────────────────────────────────────────────────────────
import os, shutil

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

files_to_save = [
    ('/content/models/ecg_multilabel_v2.pt', f'{DRIVE_MODELS}/ecg_multilabel_v2.pt'),
]

for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Saved: {dst} ({size_mb:.1f} MB)')
    else:
        print(f'WARNING: {src} not found - training may have failed')

print('\nDone! Download ecg_multilabel_v2.pt from Drive and place it in models/ locally.')